# OceanEmbed Demo
PS-26066 | MoES/INCOIS | SIH 2026

In [ ]:
import sys
sys.path.insert(0, "../model")
sys.path.insert(0, "../acoustic")
import torch
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print("Setup OK")

## SVP / Acoustic Layer

In [ ]:
from svp import compute_svp_profile
from sld import compute_sld
from sofar import compute_sofar_axis

depths=[0,5,10,20,30,50,75,100,125,150,200,300,500,700,1000]
temps=[28,27,26,25,24,22,20,18,16,14,12,10,8,6,4]
sals=[35]*15

svp=compute_svp_profile(temps,sals,depths)
sld_d,sld_v=compute_sld(svp,depths)
sofar_d,sofar_v=compute_sofar_axis(svp,depths)
print(f"SLD: {sld_d}m at {sld_v:.2f} m/s")
print(f"SOFAR: {sofar_d}m at {sofar_v:.2f} m/s")

In [ ]:
plt.figure(figsize=(5,8))
plt.plot(svp,depths,"b-o",markersize=4)
plt.axhline(sld_d,color="red",linestyle="--",label=f"SLD {sld_d}m")
plt.axhline(sofar_d,color="green",linestyle="--",label=f"SOFAR {sofar_d}m")
plt.gca().invert_yaxis()
plt.xlabel("Sound Velocity (m/s)")
plt.ylabel("Depth (m)")
plt.title("Sound Velocity Profile")
plt.legend()
plt.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig("svp.png",dpi=120)
print("SVP plot saved")

## Model Inference

In [ ]:
from encoder_cnn import CNNEncoder
from encoder_vit import ViTEncoder
from decoder import DualEncoderWithDecoder
from thermocline_head import ThermoclineHead

cnn=CNNEncoder(in_channels=5,embedding_dim=512)
vit=ViTEncoder(in_channels=5,embedding_dim=512)
model=DualEncoderWithDecoder(cnn,vit,num_depths=15)
thermo=ThermoclineHead(num_depths=15)

ckpt="../checkpoints/best_model.pt"
if os.path.exists(ckpt):
    c=torch.load(ckpt,map_location="cpu")
    model.load_state_dict(c["model"])
    thermo.load_state_dict(c["thermo"])
    print("Checkpoint loaded")
else:
    print("Demo mode - random weights")
model.eval()
thermo.eval()

In [ ]:
patch=torch.ones(1,5,64,64)*28.0
with torch.no_grad():
    T_pred=model(patch)
    td,tg=thermo(T_pred)
T_vals=T_pred.squeeze().tolist()
for d,t in zip(depths,T_vals):
    print(f"  {d:4d}m: {t:.2f} C")
print(f"Thermocline: {td.item():.1f}m")

## Summary
| Component | Status |
|---|---|
| Pipeline | Done |
| Acoustic | Done |
| CNN+ViT Encoder | Done |
| Decoder | Done |
| FastAPI | Done |
| React Frontend | Done |
| Tests 9/9 | Passing |